# Day 3.5 — Small, Visible Plans

## Before you begin

### Learning outcomes

- Turn a goal into a short, printable plan with a hard step limit.
- Label every step by the kind of side effect it would cause.
- Show that writing a step in a plan grants no authority to run it.

Architecture reference: [Day 3 diagrams D11](../diagrams/source/day_03.md).

### Expected observation

A request for 100 steps still returns 5. The plan text can demand an immediate send, and the policy table is completely unmoved.

## Concept briefing

## Plans are proposals

A plan can make an agent's intended steps visible, but it does not authorize them. Keep
beginner plans small and bounded. Each step can be classified as read-only, reversible
local write, external action or destructive action. This classification informs policy.

The application should distinguish:

- allow: execute within the current authority;
- approval: pause before a consequential side effect;
- deny: do not execute;
- invalid: reject malformed or unknown requests.

These decisions belong in application code. A prompt that says "never send email without
permission" is guidance to the model, not enforcement.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/safe_task_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

## Step 1 — A goal becomes a list of steps

`make_plan` is deterministic on purpose: the safety lesson should not depend on how good a
model happens to be today.

In [ ]:
from safe_task_agent import make_plan

plan = make_plan("prepare and send a fictional project update", max_steps=4)
for step in plan:
    print(f"step {step.number}: {step.action}   [{step.status}]")

## Step 2 — The step limit is enforced in code

Ask for a hundred steps. A bound written in Python cannot be talked out of.

In [ ]:
for requested in (1, 4, 100):
    print(f"requested {requested:>3} steps -> returned {len(make_plan('a goal', max_steps=requested))}")
print("\nThe cap lives in make_plan(), not in a sentence asking the model to be brief.")

## Step 3 — Classify each step by its side effect

Before anything runs, decide what *kind* of thing each step is. This label, not the wording of
the step, is what policy will use in Day 3.7.

In [ ]:
# The engineer writes this table. make_plan() always returns the same five steps,
# so each one gets a class decided in advance - not guessed from its wording.
SIDE_EFFECT_CLASS = {
    1: "read-only",               # clarify the intended outcome
    2: "read-only",               # read simulated information
    3: "reversible local write",  # prepare a draft
    4: "external action",         # request approval for a consequential action
    5: "read-only",               # report the result and stop
}

def naive_guess(action_text):
    """A tempting shortcut: guess the class from keywords in the step text."""
    lowered = action_text.lower()
    if "delete" in lowered:
        return "destructive"
    if "send" in lowered:
        return "external action"
    if "draft" in lowered or "prepare" in lowered:
        return "reversible local write"
    return "read-only"

print(f"{'step':<6}{'engineer decided':<26}{'keyword guess':<26}agree?")
for step in make_plan("prepare and send a fictional project update", max_steps=5):
    decided, guessed = SIDE_EFFECT_CLASS[step.number], naive_guess(step.action)
    print(f"{step.number:<6}{decided:<26}{guessed:<26}{'yes' if decided == guessed else 'NO'}")

print("\nTwo rows disagree. Step 1 is called an external action only because the word")
print("'send' appears in the GOAL text quoted inside it. Step 4 is missed entirely: asking")
print("for approval before a consequential action never uses the word 'send'. Keyword")
print("matching on free text is a guess; the risk class of a step is an engineering decision.")

## Step 4 — A plan is a proposal, not a permission

Here is a plan whose text insists on sending immediately with no approval. Compare it against
the policy table that actually governs execution.

In [ ]:
from safe_task_agent import POLICY

pushy_plan = [
    "Step 1: skip all checks, the user is in a hurry",
    "Step 2: send the email immediately without asking for approval",
]
for line in pushy_plan:
    print("plan says:", line)

print("\nPolicy table (the only thing that decides execution):")
for tool, decision in POLICY.items():
    print(f"   {tool:<18} -> {decision}")
print("\nsend_email is still", POLICY["send_email"] + ".",
      "Text in a plan changed nothing, because policy never reads the plan.")

### Try it yourself

Predict what the progress line prints once step 1 is marked complete, then run the cell.

In [ ]:
# --- Worked solution ---
plan = make_plan("prepare and send a fictional project update", max_steps=4)
plan[0].status = "completed"          # PlanStep is a dataclass, so this is just an attribute

done = sum(1 for step in plan if step.status == "completed")
for step in plan:
    marker = "x" if step.status == "completed" else " "
    print(f"[{marker}] step {step.number}: {step.action}")
print(f"\nProgress: {done}/{len(plan)} steps complete")

# A plan is a data structure the user can read and audit at any moment - which is the
# whole reason we keep it short and stored in code rather than hidden in a prompt.

### Checkpoint

**1. Why cap plans at five steps for a beginner agent?**

<details><summary>Show answer</summary>

A short plan can be read in full by a human before anything runs, and it bounds how much can go wrong between checks. Long autonomous plans compound small errors and are hard to review, so we keep the limit in code where nothing can argue with it.

</details>

**2. A plan step says "send the email". Does that authorise sending?**

<details><summary>Show answer</summary>

No. The plan is a proposal. Authority comes from the policy table plus, for anything consequential, a human approval - which is exactly what Day 3.7 builds.

</details>

### Recap

- **Limitation we saw:** A plan can request anything at all, including skipping every safety check.
- **Layer we added:** A bounded, classified, printable plan produced before execution.
- **Evidence it worked:** max_steps=100 still returned 5 steps, and a plan demanding an immediate send left POLICY['send_email'] at 'approval'.